# 🌍 Project 31 — Detect Seasonal Water Changes

## 1. Introduce the Project

### Goal

In this project, we will track how the water area of a lake changes over **12 months**.

Instead of comparing only two dates, as we did in the flood mapping project, we will analyze the lake month by month.

Our main question is:

> **How does the water area change throughout the year?**

The final output will include:

- Monthly water maps
- Monthly water area calculations
- A chart showing seasonal water changes

---

## Why Is This Useful in Remote Sensing?

Lakes and reservoirs do not always have the same water extent throughout the year.

Their area may change because of:

- Rainfall
- Drought
- Snowmelt
- Evaporation
- Dam management
- Irrigation
- Seasonal climate patterns

Remote sensing allows us to monitor these changes without physically measuring the entire lake every month.

### Real-world applications

- Drought monitoring
- Water resource management
- Hydrology
- Climate change research
- Reservoir monitoring

---

## Main New Skills

In this project, you will learn:

- Monthly `ImageCollection`s
- Repeating a workflow for multiple months
- Area calculations for each month
- Creating an `ee.FeatureCollection`
- Creating a chart from calculated values

---

# 2. Big Picture

Previously, in Project 11, we measured the area of water for one image.

Now we repeat that idea 12 times.

```text
January Images
      ↓
Monthly Composite
      ↓
NDWI
      ↓
Water Mask
      ↓
Calculate Area
      ↓

February Images
      ↓
Monthly Composite
      ↓
NDWI
      ↓
Water Mask
      ↓
Calculate Area
      ↓

      ...

December
      ↓

Monthly Water Areas
        ↓
      Chart
````

The main idea is:

```text
One Lake
   ↓
12 Months
   ↓
12 Water Area Measurements
   ↓
Seasonal Water Change
```

---

# 3. Explain the Main Concept

## Monthly ImageCollections

Previously, we used:

```javascript
collection.filterDate(startDate, endDate)
```

to select images from a specific period.

Now we will repeat this for every month.

For example:

```text
January
Jan 1 → Feb 1

February
Feb 1 → Mar 1

March
Mar 1 → Apr 1
```

For each month:

```text
Sentinel-2 ImageCollection
        ↓
Filter by Month
        ↓
Cloud Mask
        ↓
median()
        ↓
Monthly Composite
```

So even though there may be multiple satellite images during January, we create one representative image for that month.

---

# 4. Connection to Previous Lessons

This project combines several things you already know.

### From Project 9 — NDWI

We use:

```javascript
image.normalizedDifference(["B3", "B8"])
```

to detect water.

---

### From Project 11 — Measure Lake Area

We use:

```javascript
ee.Image.pixelArea()
```

and:

```javascript
reduceRegion()
```

to calculate water area.

---

### From Project 16 — NDVI Time Series

Previously:

```text
Multiple Dates
      ↓
Mean NDVI
      ↓
Chart
```

Now:

```text
Multiple Months
      ↓
Water Area
      ↓
Chart
```

So the concept is similar, but instead of tracking vegetation values, we track **lake area**.

---

# 5. Step 1 — Define the Study Area

For this project, your `studyArea` should represent the lake you want to monitor.

You can draw the lake area in the GEE Code Editor and import it as:

```javascript
var studyArea = geometry;
```

For example:

```text
Lake Boundary
      ↓
studyArea
      ↓
Filter Satellite Images
      ↓
Calculate Water Area
```

It is important that the study area covers the lake and its surrounding shoreline.

---

# 6. Step 2 — Load the Sentinel-2 Collection

As agreed, we use:

```javascript
"COPERNICUS/S2_SR_HARMONIZED"
```

Let's select images for the entire year.

```javascript
var collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(studyArea)
  .filterDate("2025-01-01", "2026-01-01");
```

### What does this return?

The result is:

```text
ee.ImageCollection
```

containing Sentinel-2 images from the selected year.

At this stage:

```text
Many Images
```

not one image.

Later, we will create a separate monthly composite.

---

# 7. Step 3 — Create a Cloud Masking Function

We learned earlier that clouds can cause problems in satellite analysis.

For this example, we use the Sentinel-2 Scene Classification Layer (`SCL`) to remove several unwanted pixels.

```javascript
function maskS2(image) {
  
  var scl = image.select("SCL");
  
  var mask = scl.neq(3)
    .and(scl.neq(8))
    .and(scl.neq(9))
    .and(scl.neq(10));
  
  return image.updateMask(mask);
}
```

### What does this do?

The `SCL` band classifies different pixel types.

Here, we remove classes associated with:

* Cloud shadows
* Medium-probability clouds
* High-probability clouds
* Cirrus clouds

The important idea is:

```text
Original Image
      ↓
Cloud Mask
      ↓
Cleaner Image
```

We will apply this function to every image.

```javascript
var cleanCollection = collection.map(maskS2);
```

---

## What Does `.map()` Do Here?

You already learned `.map()` in Project 27.

The idea is:

```text
Image 1 → maskS2()
Image 2 → maskS2()
Image 3 → maskS2()
Image 4 → maskS2()

        ↓

Clean ImageCollection
```

So:

```javascript
collection.map(maskS2)
```

applies the same function to every image.

The result is still an:

```text
ee.ImageCollection
```

---

# 8. Step 4 — Create Monthly Composites

Now we need to repeat the process for 12 months.

First, create a list:

```javascript
var months = ee.List.sequence(1, 12);
```

This creates:

```text
1, 2, 3, 4, ... 12
```

Each number represents a month.

Now we will create a function:

```javascript
function calculateMonthlyArea(month) {
```

This function will perform the same workflow for every month.

Inside the function, we first create the starting date:

```javascript
var start = ee.Date.fromYMD(2025, month, 1);
```

For example:

```text
month = 1

↓

2025-01-01
```

Then:

```javascript
var end = start.advance(1, "month");
```

This creates:

```text
Start: January 1

↓

Advance 1 month

↓

End: February 1
```

---

# 9. Step 5 — Filter Images for One Month

Inside the function:

```javascript
var monthlyCollection = cleanCollection
  .filterDate(start, end);
```

Now we have:

```text
Full Year Collection
        ↓
Filter January
        ↓
January ImageCollection
```

For February, the same function will create:

```text
February ImageCollection
```

And so on.

---

# 10. Step 6 — Create a Monthly Composite

A month may contain several Sentinel-2 images.

So we use:

```javascript
var monthlyImage = monthlyCollection
  .median()
  .clip(studyArea);
```

Remember from Project 22:

```text
Many Images
     ↓
median()
     ↓
One Composite
```

The result is:

```text
ee.Image
```

This is important because NDWI and pixel-level calculations work on an `ee.Image`.

---

# 11. Step 7 — Calculate NDWI

Now we calculate NDWI for the monthly composite.

```javascript
var ndwi = monthlyImage
  .normalizedDifference(["B3", "B8"])
  .rename("NDWI");
```

Again:

```text
B3 = Green

B8 = NIR
```

Formula:

```text
NDWI = (Green - NIR)
       -------------
       (Green + NIR)
```

---

# 12. Step 8 — Create a Water Mask

We now apply a threshold.

```javascript
var water = ndwi.gt(0.2);
```

This means:

```text
NDWI > 0.2
     ↓
Water
```

The result is approximately:

```text
Water = 1

Non-water = 0
```

Remember:

> `0.2` is only an educational example. A real research project should test and justify the threshold.

---

# 13. Step 9 — Calculate Water Area

Now we use the same idea from Project 11.

First:

```javascript
var waterAreaImage = water
  .selfMask()
  .multiply(ee.Image.pixelArea());
```

Let's break this down.

### `water.selfMask()`

This removes the non-water pixels from the calculation.

Conceptually:

```text
Water pixels
    ↓
Keep

Non-water pixels
    ↓
Mask
```

---

### `ee.Image.pixelArea()`

Every pixel has an area.

For Sentinel-2 at 10 m resolution:

```text
10 m × 10 m

↓

100 m²
```

Conceptually:

```text
Water Mask
    ×
Pixel Area
    ↓
Water Area Image
```

---

Now we sum the area:

```javascript
var area = waterAreaImage.reduceRegion({
  reducer: ee.Reducer.sum(),
  geometry: studyArea,
  scale: 10,
  maxPixels: 1e9
});
```

### What does this do?

```text
Every Water Pixel Area

        ↓

Sum

        ↓

Total Water Area
```

The result is a dictionary-like object containing the total area.

We retrieve the value:

```javascript
var areaM2 = ee.Number(
  area.get("NDWI")
);
```

Then convert square meters to square kilometers:

```javascript
var areaKm2 = areaM2.divide(1e6);
```

Because:

```text
1 km² = 1,000,000 m²
```

---

# 14. Step 10 — Store the Monthly Result

We need to save:

* The month
* The calculated water area

For this, we create an `ee.Feature`.

```javascript
return ee.Feature(null, {
  month: month,
  water_area_km2: areaKm2
});
```

This creates one record conceptually like:

```text
Month: 1
Water Area: 125 km²
```

For February:

```text
Month: 2
Water Area: 130 km²
```

Eventually:

```text
Month 1 → Area
Month 2 → Area
Month 3 → Area
...
Month 12 → Area
```

---

# 15. Step 11 — Repeat for All 12 Months

Now we apply the function to every month.

```javascript
var monthlyAreas = ee.FeatureCollection(
  months.map(calculateMonthlyArea)
);
```

The workflow is:

```text
Months

1
2
3
...
12

 ↓

calculateMonthlyArea()

 ↓

Feature 1
Feature 2
Feature 3
...
Feature 12

 ↓

FeatureCollection
```

This is the new important concept.

## What Is an `ee.FeatureCollection` Here?

It is a collection of monthly records.

Conceptually:

```text
FeatureCollection

├── January → 120 km²
├── February → 135 km²
├── March → 150 km²
├── April → 145 km²
│
└── ...
```

---

# 16. Step 12 — Create a Seasonal Water Area Chart

Now we create a chart.

```javascript
var chart = ui.Chart.feature.byFeature({
  features: monthlyAreas,
  xProperty: "month",
  yProperties: ["water_area_km2"]
})
.setChartType("LineChart")
.setOptions({
  title: "Monthly Lake Water Area",
  hAxis: {
    title: "Month"
  },
  vAxis: {
    title: "Water Area (km²)"
  },
  lineWidth: 2,
  pointSize: 4
});
```

Then:

```javascript
print(chart);
```

The result will look conceptually like:

```text
Water Area
    ↑
    |
    |        ●
    |     ●     ●
    |  ●           ●
    |                 ●
    |________________________→
      Jan Feb Mar Apr ... Dec
```

Now we can observe:

* Increasing water area
* Decreasing water area
* Seasonal peaks
* Seasonal lows

---

# 17. Complete Workflow

```text
Sentinel-2 Collection
        ↓
Filter One Year
        ↓
Cloud Masking
        ↓
Months 1 → 12
        ↓
For Each Month:
        ↓
Filter Date
        ↓
Monthly ImageCollection
        ↓
median()
        ↓
Monthly Image
        ↓
NDWI
        ↓
Water Threshold
        ↓
Water Mask
        ↓
pixelArea()
        ↓
reduceRegion(sum)
        ↓
Water Area
        ↓
FeatureCollection
        ↓
Line Chart
```

---

# Complete Code

```javascript
// ========================================
// 1. Study Area
// ========================================

// Draw or import your lake boundary in the GEE Code Editor
var studyArea = geometry;


// ========================================
// 2. Sentinel-2 Collection
// ========================================

var collection = ee.ImageCollection(
    "COPERNICUS/S2_SR_HARMONIZED"
  )
  .filterBounds(studyArea)
  .filterDate("2025-01-01", "2026-01-01");


// ========================================
// 3. Cloud Masking Function
// ========================================

function maskS2(image) {
  
  var scl = image.select("SCL");
  
  var mask = scl.neq(3)
    .and(scl.neq(8))
    .and(scl.neq(9))
    .and(scl.neq(10));
  
  return image.updateMask(mask);
}


// Apply cloud masking

var cleanCollection = collection.map(maskS2);


// ========================================
// 4. Create List of Months
// ========================================

var months = ee.List.sequence(1, 12);


// ========================================
// 5. Calculate Monthly Water Area
// ========================================

function calculateMonthlyArea(month) {
  
  // Create start and end dates
  
  var start = ee.Date.fromYMD(2025, month, 1);
  
  var end = start.advance(1, "month");
  
  
  // Filter images for one month
  
  var monthlyCollection = cleanCollection
    .filterDate(start, end);
  
  
  // Create monthly composite
  
  var monthlyImage = monthlyCollection
    .median()
    .clip(studyArea);
  
  
  // Calculate NDWI
  
  var ndwi = monthlyImage
    .normalizedDifference(["B3", "B8"])
    .rename("NDWI");
  
  
  // Water threshold
  
  var water = ndwi.gt(0.2);
  
  
  // Calculate water area
  
  var waterAreaImage = water
    .selfMask()
    .multiply(ee.Image.pixelArea());
  
  
  var area = waterAreaImage.reduceRegion({
    
    reducer: ee.Reducer.sum(),
    
    geometry: studyArea,
    
    scale: 10,
    
    maxPixels: 1e9
    
  });
  
  
  // Get area in square meters
  
  var areaM2 = ee.Number(
    area.get("NDWI")
  );
  
  
  // Convert to square kilometers
  
  var areaKm2 = areaM2.divide(1e6);
  
  
  // Return monthly result
  
  return ee.Feature(null, {
    
    month: month,
    
    water_area_km2: areaKm2
    
  });
}


// ========================================
// 6. Calculate All Monthly Areas
// ========================================

var monthlyAreas = ee.FeatureCollection(
  months.map(calculateMonthlyArea)
);


// Print results

print("Monthly Water Areas", monthlyAreas);


// ========================================
// 7. Create Chart
// ========================================

var chart = ui.Chart.feature.byFeature({
  
  features: monthlyAreas,
  
  xProperty: "month",
  
  yProperties: ["water_area_km2"]
  
})
.setChartType("LineChart")
.setOptions({
  
  title: "Monthly Lake Water Area",
  
  hAxis: {
    title: "Month"
  },
  
  vAxis: {
    title: "Water Area (km²)"
  },
  
  lineWidth: 2,
  
  pointSize: 4
  
});


// Print chart

print(chart);


// ========================================
// 8. Display Study Area
// ========================================

Map.centerObject(studyArea, 10);

Map.addLayer(
  studyArea,
  {},
  "Study Area"
);
```

---

# Things to Remember

## 1. `.filterDate()` Returns an ImageCollection

```javascript
cleanCollection.filterDate(start, end)
```

Result:

```text
ee.ImageCollection
```

It may contain multiple images.

---

## 2. `.median()` Converts Multiple Images Into One Composite

```text
ImageCollection
       ↓
    median()
       ↓
ee.Image
```

This allows us to calculate NDWI for one representative image each month.

---

## 3. `.map()` Can Repeat the Same Process

We used:

```javascript
months.map(calculateMonthlyArea)
```

Meaning:

```text
January → Calculate Area

February → Calculate Area

March → Calculate Area

...

December → Calculate Area
```

Instead of writing the same code 12 times, we use a function.

---

## 4. `ee.Feature` Stores One Result

Example:

```text
Month: 5

Water Area: 210 km²
```

One record = one `Feature`.

---

## 5. `ee.FeatureCollection` Stores Multiple Results

```text
January
February
March
...
December
```

These results can then be used to create charts.

---

## 6. The NDWI Threshold Is Not Universal

We used:

```javascript
ndwi.gt(0.2)
```

But this does **not** mean `0.2` is always correct.

The best threshold may depend on:

* The lake
* Water turbidity
* Surrounding land
* Shadows
* Season
* Atmospheric conditions

For a research-grade analysis, the threshold should be evaluated and justified.

---

# Common Beginner Mistakes

## ❌ Mistake 1: Calculating the Area From the Entire ImageCollection

Incorrect idea:

```text
ImageCollection
      ↓
reduceRegion()
```

`reduceRegion()` is generally used with an image.

Correct workflow:

```text
ImageCollection
      ↓
median()
      ↓
Image
      ↓
NDWI
      ↓
Water Mask
      ↓
Area Calculation
```

---

## ❌ Mistake 2: Forgetting That Each Month Needs Its Own Date Range

Incorrect:

```text
Whole Year
    ↓
One Composite
    ↓
One Area
```

This only gives one result for the entire year.

Correct:

```text
January → Area

February → Area

March → Area

...

December → Area
```

---

## ❌ Mistake 3: Assuming Every Change Is Caused by Climate

If the lake area decreases:

```text
Water Area ↓
```

this does not automatically mean:

```text
Climate Change
```

Other causes may include:

* Water extraction
* Dam operation
* Irrigation
* Seasonal evaporation
* Reduced rainfall
* Natural seasonal cycles

Remote sensing shows the change.

Explaining **why** the change happened requires additional information.

---

## ❌ Mistake 4: Using a Threshold Without Checking the Result

For example:

```javascript
ndwi.gt(0.2)
```

Always visually inspect the water mask.

A poor threshold can:

* Miss shallow water
* Include wet soil
* Include shadows
* Incorrectly classify land as water

---

## ❌ Mistake 5: Ignoring Months With Poor Imagery

Some months may have:

* Heavy cloud cover
* Very few images
* Missing observations

A monthly composite is not automatically perfect.

For a research project, you should check:

```text
How many usable images exist each month?
```

---

## ❌ Mistake 6: Confusing Lake Area With Water Volume

This project measures:

```text
Surface Area
```

not:

```text
Water Volume
```

A lake can lose water depth without a large change in surface area.

To estimate volume, you would need additional information such as:

* Water level
* Bathymetry
* Elevation relationships

---

# Your Practice — Project 31.1

## Seasonal Water Monitoring for Another Lake

Choose a different lake or reservoir.

Your task is to monitor its water area over **one full year**.

### Requirements

Your project should:

1. Define a lake study area.
2. Use:

```javascript
"COPERNICUS/S2_SR_HARMONIZED"
```

3. Select one full year of Sentinel-2 imagery.
4. Create monthly `ImageCollection`s.
5. Create a monthly composite using:

```javascript
.median()
```

6. Calculate NDWI for every month.
7. Create a water mask using a threshold.
8. Calculate water area using:

```javascript
ee.Image.pixelArea()
```

9. Convert the area to:

```text
km²
```

10. Store the results for all 12 months.
11. Create a chart showing monthly water area.

---

## What You Should Produce

```text
Lake
  ↓
12 Monthly Water Measurements
  ↓
FeatureCollection
  ↓
Water Area Chart
```

Try to inspect your final chart and answer:

> During which month was the lake area largest?

> During which month was it smallest?

> Does the pattern appear seasonal?

Do not worry if your first version has errors. The goal is to practice the workflow.

---

# Conceptual Questions

### Question 1

Why do we use `.median()` before calculating NDWI instead of simply selecting one random Sentinel-2 image each month?

---

### Question 2

What is the difference between these two workflows?

```text
One Year
   ↓
median()
   ↓
One Water Area
```

and:

```text
12 Individual Months
        ↓
12 Monthly Composites
        ↓
12 Water Areas
```

What information does the second workflow provide that the first one does not?

---

### Question 3

Suppose your water area chart looks like this:

```text
January     → 100 km²

April       → 180 km²

August      → 90 km²

December    → 160 km²
```

Can you conclude that the lake is permanently shrinking?

Why or why not?
